In [2]:
# Import required libs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
dataset = pd.read_csv("Social_Network_Ads.csv")

In [4]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [5]:
dataset = pd.get_dummies(dataset, dtype=int, drop_first = True)

In [6]:
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [7]:
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [8]:
# Since the User_ID column doesn't give meaningful info, we drop the data in rows (axis = 1 means row)
dataset = dataset.drop("User ID", axis=1)
dataset

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [9]:
# Check the values counts. It will help to check the dataset balanced or not
dataset["Purchased"].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [10]:
X = dataset[["Age", "EstimatedSalary", "Gender_Male"]]
y = dataset["Purchased"]

In [11]:
X.shape

(400, 3)

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

In [13]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

params = {
        "criterion": ['gini', 'entropy', 'log_loss'],
        "n_estimators": [50, 100, 150]
       }

grid = GridSearchCV(RandomForestClassifier(), params, refit=True, verbose=3, n_jobs=-1, scoring="f1_weighted")
grid.fit(X_train, y_train)

Fitting 5 folds for each of 9 candidates, totalling 45 fits


,estimator,RandomForestClassifier()
,param_grid,"{'criterion': ['gini', 'entropy', ...], 'n_estimators': [50, 100, ...]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,150


In [15]:
results = grid.cv_results_
print(results)

{'mean_fit_time': array([0.16074495, 0.28950577, 0.42896376, 0.14517031, 0.27204599,
       0.42347975, 0.13840046, 0.27318387, 0.39453812]), 'std_fit_time': array([0.00813505, 0.00699801, 0.00669282, 0.00769764, 0.01832032,
       0.00630901, 0.00685954, 0.00576249, 0.01190849]), 'mean_score_time': array([0.01153088, 0.01489477, 0.02018857, 0.00993943, 0.01404366,
       0.01813431, 0.01038942, 0.01369028, 0.01608334]), 'std_score_time': array([0.0006088 , 0.00178149, 0.00055873, 0.00109184, 0.00176349,
       0.00211427, 0.00038354, 0.00113852, 0.00361106]), 'param_criterion': masked_array(data=['gini', 'gini', 'gini', 'entropy', 'entropy',
                   'entropy', 'log_loss', 'log_loss', 'log_loss'],
             mask=[False, False, False, False, False, False, False, False,
                   False],
       fill_value=np.str_('?'),
            dtype=object), 'param_n_estimators': masked_array(data=[50, 100, 150, 50, 100, 150, 50, 100, 150],
             mask=[False, False, Fals

In [16]:
grid_pred = grid.predict(X_test)
grid_pred

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 1, 0, 0, 1])

In [17]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [18]:
matrix = confusion_matrix(y_test, grid_pred)

In [19]:
print(matrix)

[[74  5]
 [ 5 36]]


In [20]:
report = classification_report(y_test, grid_pred)
print(report)

              precision    recall  f1-score   support

           0       0.94      0.94      0.94        79
           1       0.88      0.88      0.88        41

    accuracy                           0.92       120
   macro avg       0.91      0.91      0.91       120
weighted avg       0.92      0.92      0.92       120



In [21]:
from sklearn.metrics import f1_score
f1_macro = f1_score(y_test, grid_pred, average="weighted")
print("The f1_macro value for best parameter {}:".format(grid.best_params_), f1_macro)

The f1_macro value for best parameter {'criterion': 'entropy', 'n_estimators': 150}: 0.9166666666666666


In [22]:
from sklearn.metrics import roc_auc_score

# It will not work, if not provide the probability data (predict_proba)
roc_auc_score(y_test, grid.predict_proba(X_test)[:,1])

0.9643408459401049

In [23]:
table = pd.DataFrame.from_dict(results)

In [24]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.160745,0.008135,0.011531,0.000609,gini,50,"{'criterion': 'gini', 'n_estimators': 50}",0.874254,0.838326,0.806153,0.892857,0.946153,0.871549,0.047765,9
1,0.289506,0.006998,0.014895,0.001781,gini,100,"{'criterion': 'gini', 'n_estimators': 100}",0.874254,0.838326,0.823129,0.911105,0.946153,0.878593,0.045471,7
2,0.428964,0.006693,0.020189,0.000559,gini,150,"{'criterion': 'gini', 'n_estimators': 150}",0.874254,0.875644,0.823129,0.911105,0.963889,0.889604,0.046540,2
3,0.145170,0.007698,0.009939,0.001092,entropy,50,"{'criterion': 'entropy', 'n_estimators': 50}",0.874254,0.857143,0.823129,0.911105,0.963889,0.885904,0.048209,5
4,0.272046,0.018320,0.014044,0.001763,entropy,100,"{'criterion': 'entropy', 'n_estimators': 100}",0.874254,0.821429,0.823129,0.911105,0.946153,0.875214,0.048841,8
5,0.423480,0.006309,0.018134,0.002114,entropy,150,"{'criterion': 'entropy', 'n_estimators': 150}",0.892857,0.875644,0.859435,0.911105,0.963889,0.900586,0.036037,1
6,0.138400,0.006860,0.010389,0.000384,log_loss,50,"{'criterion': 'log_loss', 'n_estimators': 50}",0.892857,0.858503,0.823129,0.911105,0.909115,0.878942,0.033679,6
7,0.273184,0.005762,0.013690,0.001139,log_loss,100,"{'criterion': 'log_loss', 'n_estimators': 100}",0.874254,0.840114,0.841398,0.911105,0.963889,0.886152,0.046754,3
8,0.394538,0.011908,0.016083,0.003611,log_loss,150,"{'criterion': 'log_loss', 'n_estimators': 150}",0.874254,0.857143,0.859435,0.911105,0.927778,0.885943,0.028468,4


In [26]:
age = int(input("Age: "))
gender = int(input("Gender [Male=0, Female=1]: "))
est_salary = float(input("Est. Salary: "))

prediction = grid.predict([[age, gender, est_salary]])

prediction = "seems not purchase" if gender == 0 else "possible to purchase"
gender = "male" if gender == 0 else "female"

print("Using RandomForestClassifier: {} years old {} getting {} salary, is {}".format(age, gender, est_salary, prediction))

Age:  42
Gender [Male=0, Female=1]:  0
Est. Salary:  200000


Using RandomForestClassifier: 42 years old male getting 200000.0 salary, is seems not purchase
